In [32]:
import pandas as pd
import numpy as np
import sqlite3

conn = sqlite3.connect('../diabetes-readmission.db')
df = pd.read_sql("""
    SELECT * FROM encounters_clean
""", conn)
conn.close()

df.head()

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,readmitted,any_med_change,...,A1Cresult_Unknown,admission_type_Emergency,admission_type_Other,admission_type_Urgent,admission_source_Emergency Room,admission_source_Other,admission_source_Physician Referral,admission_source_Transfer from a Skilled Nursing Facility (SNF),admission_source_Transfer from a hospital,admission_source_Transfer from another health care facility
0,1,41,0,1,0.000000,0.0,0.000000,1,0,0,...,1,0,1,0,0,0,1,0,0,0
1,3,59,0,18,0.000000,0.0,0.000000,9,1,1,...,1,1,0,0,1,0,0,0,0,0
2,2,11,5,13,1.098612,0.0,0.693147,6,0,0,...,1,1,0,0,1,0,0,0,0,0
3,2,44,1,16,0.000000,0.0,0.000000,7,0,1,...,1,1,0,0,1,0,0,0,0,0
4,1,51,0,8,0.000000,0.0,0.000000,5,0,0,...,1,1,0,0,1,0,0,0,0,0


In [33]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

cols_to_transform = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

X = df.drop(columns='readmitted')
y = df['readmitted']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

scaler = RobustScaler()

X_train[cols_to_transform] = scaler.fit_transform(X_train[cols_to_transform])
X_test[cols_to_transform] = scaler.transform(X_test[cols_to_transform])

X_train.columns = X_train.columns.str.replace('[', '', regex=False).str.replace(']', '', regex=False).str.replace('(', '', regex=False).str.replace(')', '', regex=False).str.replace('<', '', regex=False)
X_test.columns = X_test.columns.str.replace('[', '', regex=False).str.replace(']', '', regex=False).str.replace('(', '', regex=False).str.replace(')', '', regex=False).str.replace('<', '', regex=False)

In [34]:
conn = sqlite3.connect('../diabetes-readmission.db')

X_train.to_sql("X_train", conn, if_exists="replace", index=False)
X_test.to_sql("X_test", conn, if_exists="replace", index=False)
y_train.to_sql("y_train", conn, if_exists="replace", index=False)
y_test.to_sql("y_test", conn, if_exists="replace", index=False)

conn.close()